<a href="https://colab.research.google.com/github/ekonishi8524/my-data-science-projects/blob/main/01_corona_and_nyc_taxis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import auth
import pandas as pd

auth.authenticate_user()
print("Authentication successful!")

project_id = "YOUR_GOOGLE_CLOUD_PROJECT_ID"

query_1_1 = """
SELECT
  DATE(pickup_datetime) AS ride_date,
  COUNT(*) AS total_trips,
  ROUND(AVG(fare_amount), 2) AS avg_fare,
  ROUND(AVG(tip_amount), 2) AS avg_tip,
  MAX(fare_amount) AS max_fare
FROM
  `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2020`
WHERE
  fare_amount > 0
  AND pickup_datetime BETWEEN '2020-01-01' AND '2020-12-31'
GROUP BY
  ride_date
ORDER BY
  ride_date ASC;
"""

df = pd.read_gbq(query_1_1, project_id=project_id, dialect="standard")

print(f"Shape of Acquired Data: {df.shape}")
df.head()

In [ ]:
df.info()

df.describe()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

plt.plot(df['ride_date'], df['total_trips'], color='blue', marker='o', markersize=2)

plt.title('New York Taxi Daily Trips in 2020')
plt.xlabel('Date')
plt.ylabel('Total Trips')
plt.grid(True)
plt.show()

In [ ]:
df['ride_date'] = pd.to_datetime(df['ride_date'])

df['day_of_week'] = df['ride_date'].dt.dayofweek

weekday_names = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri', 5: 'Sat', 6: 'Sun'}
df['day_name'] = df['day_of_week'].map(weekday_names)

summary_by_weekday = df.groupby('day_name')[['total_trips', 'avg_tip']].mean()

weekday_order = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
summary_by_weekday = summary_by_weekday.reindex(weekday_order)
print(summary_by_weekday)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

df['period'] = 'With-Corona'
df.loc[df['ride_date'] < '2020-03-22', 'period'] = 'Before-Corona'

period_summary = df.groupby('period')[['total_trips', 'avg_fare']].mean()
print(period_summary)

plt.figure(figsize=(8, 5))
sns.boxplot(x='period', y='avg_tip', data=df)
plt.title('Comparison of Average Tip Amount')
plt.ylabel('Average Tip ($)')
plt.show()

In [ ]:
df['tip_percentage'] = (df['avg_tip'] / df['avg_fare']) * 100

percentage_summary = df.groupby('period')['tip_percentage'].mean()
print("--- Tip Percentages (%) Before- and With-Corona ---")
print(percentage_summary)

import seaborn as sns
plt.figure(figsize=(10, 5))
sns.barplot(x='day_name', y='tip_percentage', hue='period', data=df, \
            order=weekday_order)
plt.title('Comparison of Tip Percentage by Weekday')
plt.ylabel('Tip Percentage (%)')
plt.show()

In [ ]:
query_1_2 = """
SELECT
  DATE(pickup_datetime) AS ride_date,
  COUNT(*) AS total_trips,
  ROUND(AVG(fare_amount), 2) AS avg_fare,
  ROUND(AVG(tip_amount), 2) AS avg_tip,
  ROUND(AVG(trip_distance), 2) AS avg_distance,
  ROUND(AVG(TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, SECOND) / 60.0), 1)\
   AS avg_duration_minutes
FROM
  `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2020`
WHERE
  fare_amount > 0
  AND trip_distance > 0
  AND dropoff_datetime > pickup_datetime
  AND TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, HOUR) < 24
  AND pickup_datetime BETWEEN '2020-01-01' AND '2020-12-31'
GROUP BY
  ride_date
ORDER BY
  ride_date ASC;
"""

df = pd.read_gbq(query_1_2, project_id=project_id, dialect="standard")

df['ride_date'] = pd.to_datetime(df['ride_date'])
df['period'] = 'With-Corona'
df.loc[df['ride_date'] < '2020-03-22', 'period'] = 'Before-Corona'

df.head()

In [ ]:
distance_summary = df.groupby('period')[['avg_distance', 'avg_duration_minutes']].mean()
print(distance_summary)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

df['ride_date'] = pd.to_datetime(df['ride_date'])

df['total_trips'] = df['total_trips'].astype(float)
df['avg_fare'] = df['avg_fare'].astype(float)
df['avg_tip'] = df['avg_tip'].astype(float)
df['avg_distance'] = df['avg_distance'].astype(float)

df['day_of_week'] = df['ride_date'].dt.dayofweek
weekday_names = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri', 5: 'Sat', 6: 'Sun'}
df['day_name'] = df['day_of_week'].map(weekday_names)

df_ml = pd.get_dummies(df, columns=['day_name', 'period'], drop_first=True)

df_ml['next_day_trips'] = df_ml['total_trips'].shift(-1)

df_ml = df_ml.dropna()

feature_cols = [col for col in df_ml.columns if 'day_name_' in col or 'period_' in col] + \
                ['total_trips', 'avg_fare', 'avg_distance']
X = df_ml[feature_cols]
y = df_ml['next_day_trips']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,\
                                                    random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)

print("=== AI Model Building Complete! ===")
print(f"Mean Absolute Error of Prediction: {int(mae):,} ")
print("\n--- Important Features Detected by AI（Impacts） ---")
for col, coef in zip(feature_cols, model.coef_):
    print(f"{col}: {coef:.2f}")